## Упражнение 04. A/B-тестирование

In [1]:
import pandas as pd
import sqlite3

In [2]:
connection = sqlite3.connect('../data/checking-logs.sqlite')

In [3]:
test_results = pd.io.sql.read_sql(
    """
    WITH user_periods AS (
        SELECT
            t.uid,
            CASE
                WHEN t.first_commit_ts >= t.first_view_ts
                    THEN 'after'
                ELSE 'before'
            END AS time,
            AVG(
                (
                    strftime('%s', t.first_commit_ts)
                    - d.deadlines
                ) / 3600
            ) AS user_avg_diff
        FROM test AS t
        JOIN deadlines AS d
            ON t.labname = d.labs
        WHERE t.labname != 'project1'
        GROUP BY t.uid, time
    ),
    valid_users AS (
        SELECT uid
        FROM user_periods
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    )
    SELECT
        time,
        AVG(user_avg_diff) AS avg_diff
    FROM user_periods
    WHERE uid IN (
        SELECT uid
        FROM valid_users
    )
    GROUP BY time
    ORDER BY time;
    """,
    connection
)

test_results

,time,avg_diff
0,after,-99.523810
1,before,-66.047619


In [4]:
control_results = pd.io.sql.read_sql(
    """
    WITH user_periods AS (
        SELECT
            c.uid,
            CASE
                WHEN c.first_commit_ts >= c.first_view_ts
                    THEN 'after'
                ELSE 'before'
            END AS time,
            AVG(
                (
                    strftime('%s', c.first_commit_ts)
                    - d.deadlines
                ) / 3600
            ) AS user_avg_diff
        FROM control AS c
        JOIN deadlines AS d
            ON c.labname = d.labs
        WHERE c.labname != 'project1'
        GROUP BY c.uid, time
    ),
    valid_users AS (
        SELECT uid
        FROM user_periods
        GROUP BY uid
        HAVING COUNT(DISTINCT time) = 2
    )
    SELECT
        time,
        AVG(user_avg_diff) AS avg_diff
    FROM user_periods
    WHERE uid IN (
        SELECT uid
        FROM valid_users
    )
    GROUP BY time
    ORDER BY time;
    """,
    connection
)

control_results

,time,avg_diff
0,after,-99.322222
1,before,-98.033333


In [5]:
connection.close()

### Вывод

Да, гипотеза может быть верна.

В тестовой группе средняя дельта изменилась примерно с −66 часов до первого посещения ленты до −100 часов после посещения. Более отрицательная дельта означает, что пользователи стали начинать работу раньше относительно дедлайна.

В контрольной группе значения практически не изменились: примерно −98 часов до временной границы и −99 часов после неё.

Следовательно, заметное изменение наблюдается в тестовой группе, но почти отсутствует в контрольной. В рамках упрощённого A/B-теста это подтверждает гипотезу о влиянии ленты новостей на поведение пользователей.